# P21: Sub-1B Model — Remaining Seeds (456, 999)
QLoRA fine-tune Qwen2.5-0.5B on AG News. Seeds 77 & 123 already done.

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate scikit-learn

In [ ]:
import torch, random, math, json, time, os, gc
from collections import Counter
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from tqdm import tqdm

# Config
SEEDS = [456, 999]  # Only remaining seeds
DOMAIN = "ag_news"
TRAIN_SIZE = 5000
TEST_SIZE = 1000
MAX_LEN = 384
EPOCHS = 3
LR = 2e-4
BATCH = 4
GRAD_ACC = 8
LORA_R = 64
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
RD = "/kaggle/working/results_p21"
os.makedirs(RD, exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Seeds to run: {SEEDS}")
print(f"Results dir: {RD}")

In [ ]:
# Load dataset once
print("Loading AG News...")
ds = load_dataset("ag_news")
LABELS = ["World","Sports","Business","Sci/Tech"]
SYS = "Classify this news. Reply with ONLY: World, Sports, Business, or Sci/Tech"
def get(ex): return ex["text"], LABELS[ex["label"]]
print(f"Loaded: {len(ds['train'])} train, {len(ds['test'])} test")

In [ ]:
def run_seed(SEED):
    print(f"\n{'='*60}")
    print(f"=== P21 Qwen2.5-0.5B seed {SEED} ===")
    print(f"{'='*60}")
    
    random.seed(SEED)
    torch.manual_seed(SEED)
    
    # Sample data
    raw_tr = ds["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
    raw_te = ds["test"].shuffle(seed=SEED).select(range(TEST_SIZE))
    lc = Counter([get(ex)[1] for ex in raw_tr]); tot=sum(lc.values())
    H = -sum((c/tot)*math.log2(c/tot) for c in lc.values())
    print(f"{DOMAIN}: {len(LABELS)} classes, H={H:.2f}, train={len(raw_tr)}, test={len(raw_te)}")
    
    # Load model
    tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True),
        device_map="auto", trust_remote_code=True)
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_R*2,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_dropout=0, bias="none", task_type="CAUSAL_LM"))
    model.print_trainable_parameters()
    
    # Tokenize
    def tokenize(ex):
        txt, lbl = get(ex)
        msgs = [{"role":"system","content":SYS},{"role":"user","content":txt[:300]},{"role":"assistant","content":lbl}]
        s = tok.apply_chat_template(msgs, tokenize=False)
        enc = tok(s, truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
        enc["labels"] = enc["input_ids"].clone()
        enc["labels"][enc["attention_mask"]==0] = -100
        return {k:v.squeeze(0) for k,v in enc.items()}
    
    train_tok = raw_tr.map(tokenize, remove_columns=raw_tr.column_names)
    train_tok.set_format("torch")
    
    # Train
    loader = DataLoader(train_tok, batch_size=BATCH, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = (len(loader) // GRAD_ACC) * EPOCHS
    sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=10, num_training_steps=total_steps)
    
    model.train()
    t0 = time.time()
    for epoch in range(EPOCHS):
        total_loss = 0
        opt.zero_grad()
        for step, batch in enumerate(loader):
            batch = {k:v.to("cuda") for k,v in batch.items()}
            loss = model(**batch).loss / GRAD_ACC
            loss.backward()
            total_loss += loss.item() * GRAD_ACC
            if (step+1) % GRAD_ACC == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
            if (step+1) % 100 == 0:
                print(f"  E{epoch+1} step {step+1}/{len(loader)} loss={total_loss/(step+1):.4f}")
        print(f"Epoch {epoch+1}/{EPOCHS} loss={total_loss/len(loader):.4f}")
    
    train_min = (time.time()-t0)/60
    print(f"Train done ({train_min:.0f}m). Evaluating...")
    
    # Evaluate
    model.eval()
    preds, golds = [], []
    hal = 0
    for i, ex in enumerate(raw_te):
        txt, lbl = get(ex)
        msgs = [{"role":"system","content":SYS},{"role":"user","content":txt[:300]}]
        p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = tok(p, return_tensors="pt", truncation=True, max_length=MAX_LEN).to("cuda")
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=32, do_sample=False)
        resp = tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        if resp not in LABELS:
            hal += 1
        preds.append(resp); golds.append(lbl)
        if (i+1) % 200 == 0:
            print(f"    Eval {i+1}/{len(raw_te)}...")
    
    f1 = f1_score(golds, preds, average="macro", zero_division=0)
    acc = sum(p==l for p,l in zip(preds,golds))/len(golds)
    
    res = {"paper":"p21", "model":MODEL_ID, "seed":SEED,
           "f1":round(f1,4), "accuracy":round(acc,4),
           "hallucination_count":hal, "n_test":len(raw_te),
           "train_minutes":round(train_min), "domain":DOMAIN}
    
    fname = f"{RD}/p21_qwen05b_s{SEED}.json"
    with open(fname,"w") as f: json.dump(res, f, indent=2)
    
    print(f"\nDONE p21_qwen05b s{SEED}: F1={f1:.3f} acc={acc:.3f} hal={hal} ({train_min:.0f}m)")
    print(f"Saved: {fname}")
    
    # Cleanup
    del model, tok, opt, sched, loader, train_tok
    gc.collect()
    torch.cuda.empty_cache()
    
    return res

In [ ]:
# Run all remaining seeds
all_results = []
for seed in SEEDS:
    res = run_seed(seed)
    all_results.append(res)
    print(f"\n>>> Completed seed {seed}: F1={res['f1']}, Acc={res['accuracy']}")

print("\n" + "="*60)
print("ALL REMAINING P21 SEEDS DONE")
print("="*60)
for r in all_results:
    print(f"  Seed {r['seed']}: F1={r['f1']:.4f} Acc={r['accuracy']:.4f} Hal={r['hallucination_count']}")

# Show files
print(f"\nFiles in {RD}:")
for f in os.listdir(RD):
    print(f"  {f}")

In [ ]:
# Display results for easy copy
print("\n=== COPY THESE RESULTS ===")
for f in sorted(os.listdir(RD)):
    path = os.path.join(RD, f)
    with open(path) as fh:
        data = json.load(fh)
    print(f"\n--- {f} ---")
    print(json.dumps(data, indent=2))